### Decision Trees - CART

converting numerical into categories

In [1]:
import pandas as pd
import numpy as np

data = pd.DataFrame({
    'outlook': ['s','s','o','r','r','r','o','s','s','r','s','o','o','r'],
    'temp': [85,80,83,70,68,65,64,72,69,75,75,72,81,71],
    'humidity': [85,90,78,96,80,70,65,95,70,80,70,90,75,80],
    'wind': ['w','s','w','w','w','s','s','w','w','w','s','s','w','s'],
    'play': [0,0,1,1,1,0,1,0,1,1,1,1,1,0]
})

# 2. Convert numerical attributes into categories
data['temp'] = np.where(data['temp'] > 75, 'h', 'l')
data['humidity'] = np.where(data['humidity'] > 80, 'h', 'l')

features = ['outlook', 'temp', 'humidity', 'wind']
target = 'play'


def gini(y):
    p1 = np.mean(y == 1)
    p0 = np.mean(y == 0)
    return 1 - p1**2 - p0**2


def weighted_gini(df, feature):
    best_gini = float('inf')
    best_value = None

    for val in df[feature].unique():
        left = df[df[feature] == val]
        right = df[df[feature] != val]

        if len(left) == 0 or len(right) == 0:
            continue

        score = (len(left) / len(df) * gini(left[target]) + len(right) / len(df) * gini(right[target]))

        if score < best_gini:
            best_gini = score
            best_value = val

    return best_gini, best_value


def build_tree(df, features, target):

    if len(df[target].unique()) == 1:
        return df[target].iloc[0]

    if len(features) == 0:
        return df[target].mode().iloc[0]

    ginis = {}
    for feat in features :
        ginis[feat] = weighted_gini(df,feat);

    best_feature = min(ginis, key=lambda f: ginis[f][0])
    best_value = ginis[best_feature][1]

    if best_value is None:
        return df[target].mode().iloc[0]

    left = df[df[best_feature] == best_value]
    right = df[df[best_feature] != best_value]

    remaining_features = [f for f in features if f != best_feature]

    left = left.drop(columns=[best_feature])
    right = right.drop(columns=[best_feature])

    return {
        best_feature: {
            'value': best_value,
            'left': build_tree(left, remaining_features, target),
            'right': build_tree(right, remaining_features, target)
        }
    }


# 6. Classify a new sample
def classify(tree, sample, default_class):

    if not isinstance(tree, dict):
        return tree

    feature = next(iter(tree))
    node = tree[feature]

    if sample[feature] == node['value']:
        return classify(node['left'], sample, default_class)

    return classify(node['right'], sample, default_class)


from pprint import pprint
tree = build_tree(data, features, target)

print("Generated CART Decision Tree:")
pprint(tree, sort_dicts=False)


# 8. Classify a new sample
new_sample = pd.Series({
    'outlook': 's',
    'temp': 'l',
    'humidity': 'l',
    'wind': 'w'
})

default_class = data[target].mode().iloc[0]
prediction = classify(tree, new_sample, default_class)

print("Classification:", "Yes" if prediction == 1 else "No")


Generated CART Decision Tree:
{'outlook': {'value': 'o',
             'left': np.int64(1),
             'right': {'temp': {'value': 'h',
                                'left': np.int64(0),
                                'right': {'wind': {'value': 'w',
                                                   'left': {'humidity': {'value': 'h',
                                                                         'left': np.int64(0),
                                                                         'right': np.int64(1)}},
                                                   'right': np.int64(0)}}}}}}
Classification: Yes


In [ ]:
import pandas as pd
import numpy as np
from pprint import pprint

data = pd.DataFrame({
    'Experience': ['Low', 'High', 'Medium', 'Low', 'High','Medium', 'Low', 'High', 'Medium', 'Low','High', 'Medium', 'Low', 'High'],
    'Performance': ['Good', 'Good', 'Average', 'Poor', 'Average','Good', 'Average', 'Poor', 'Good', 'Poor','Good', 'Average', 'Good', 'Average'],
    'Education': ['UG', 'PG', 'UG', 'UG', 'PG','PG', 'UG', 'UG', 'PG', 'PG','UG', 'UG', 'PG', 'PG'],
    'Projects': [1, 5, 3, 1, 4, 5, 2, 2, 4, 1, 6, 3, 5, 4],
    'Promoted': ['No', 'Yes', 'Yes', 'No', 'Yes','Yes', 'No', 'No', 'Yes', 'No','Yes', 'Yes', 'Yes', 'Yes']
})

# Convert numerical attribute into categories
data['Projects'] = np.where(data['Projects'] > 3, 'h', 'l')

features = ['Experience', 'Performance', 'Education', 'Projects']
target = 'Promoted'


def gini(y):
    p1 = np.mean(y == 'Yes')
    p0 = np.mean(y == 'No')
    return 1 - p1**2 - p0**2


def weighted_gini(df, feature):
    best_gini = float('inf')
    best_value = None

    for val in df[feature].unique():
        left = df[df[feature] == val]
        right = df[df[feature] != val]

        if len(left) == 0 or len(right) == 0:
            continue

        score = ( len(left) / len(df) * gini(left[target]) + len(right) / len(df) * gini(right[target]))

        if score < best_gini:
            best_gini = score
            best_value = val

    return best_gini, best_value


def build_tree(df, features, target):

    if len(df[target].unique()) == 1:
        return df[target].iloc[0]

    if len(features) == 0:
        return df[target].mode().iloc[0]

    ginis = {}

    for feat in features:
        ginis[feat] = weighted_gini(df, feat)

    best_feature = min(ginis, key=lambda f: ginis[f][0])
    best_value = ginis[best_feature][1]

    if best_value is None:
        return df[target].mode().iloc[0]

    left = df[df[best_feature] == best_value]
    right = df[df[best_feature] != best_value]

    remaining_features = [f for f in features if f != best_feature]

    left = left.drop(columns=[best_feature])
    right = right.drop(columns=[best_feature])

    return {
        best_feature: {
            'value': best_value,
            'left': build_tree(left, remaining_features, target),
            'right': build_tree(right, remaining_features, target)
        }
    }


def classify(tree, sample, default_class):

    if not isinstance(tree, dict):
        return tree

    feature = next(iter(tree))
    node = tree[feature]

    if sample[feature] == node['value']:
        return classify(node['left'], sample, default_class)

    return classify(node['right'], sample, default_class)


tree = build_tree(data, features, target)

print("Generated CART Decision Tree:")
pprint(tree, sort_dicts=False)


new_sample = pd.Series({
    'Experience': 'Medium',
    'Performance': 'Good',
    'Education': 'PG',
    'Projects': 'h'
})

default_class = data[target].mode().iloc[0]

prediction = classify(tree, new_sample, default_class)

print("Classification:", prediction)

Generated CART Decision Tree:
{'Projects': {'value': 'l',
              'left': {'Experience': {'value': 'Medium',
                                      'left': 'Yes',
                                      'right': 'No'}},
              'right': 'Yes'}}
Classification: Yes


CART Regression

In [ ]:
import pandas as pd
import numpy as np
from pprint import pprint

data = pd.DataFrame({
    'Experience': ['Low', 'High', 'Medium', 'Low', 'High',
                   'Medium', 'Low', 'High', 'Medium', 'Low',
                   'High', 'Medium', 'Low', 'High'],

    'Performance': ['Good', 'Good', 'Average', 'Poor', 'Average',
                    'Good', 'Average', 'Poor', 'Good', 'Poor',
                    'Good', 'Average', 'Good', 'Average'],

    'Education': ['UG', 'PG', 'UG', 'UG', 'PG',
                  'PG', 'UG', 'UG', 'PG', 'PG',
                  'UG', 'UG', 'PG', 'PG'],

    'Projects': [1, 5, 3, 1, 4, 5, 2, 2, 4, 1, 6, 3, 5, 4],

    'Salary': [25000, 60000, 45000, 22000, 55000,
               50000, 30000, 35000, 52000, 24000,
               65000, 48000, 40000, 58000]
})

# Convert numerical attribute into categories
data['Projects'] = np.where(data['Projects'] > 3, 'h', 'l')

features = ['Experience', 'Performance', 'Education', 'Projects']
target = 'Salary'


def mse(y):
    return np.mean((y - np.mean(y)) ** 2)


def weighted_mse(df, feature):

    best_mse = float('inf')
    best_value = None

    for val in df[feature].unique():

        left = df[df[feature] == val]
        right = df[df[feature] != val]

        if len(left) == 0 or len(right) == 0:
            continue

        score = (len(left) / len(df) * mse(left[target]) + len(right) / len(df) * mse(right[target]))

        if score < best_mse:
            best_mse = score
            best_value = val

    return best_mse, best_value


def build_tree(df, features, target):

    if len(features) == 0:
        return df[target].mean()

    mses = {}

    for feat in features:
        mses[feat] = weighted_mse(df, feat)

    best_feature = min(mses, key=lambda f: mses[f][0])
    best_value = mses[best_feature][1]

    if best_value is None:
        return df[target].mean()

    left = df[df[best_feature] == best_value]
    right = df[df[best_feature] != best_value]

    remaining_features = [f for f in features if f != best_feature]

    left = left.drop(columns=[best_feature])
    right = right.drop(columns=[best_feature])

    return {
        best_feature: {
            'value': best_value,
            'left': build_tree(left, remaining_features, target),
            'right': build_tree(right, remaining_features, target)
        }
    }


def predict(tree, sample):

    if not isinstance(tree, dict):
        return tree

    feature = next(iter(tree))
    node = tree[feature]

    if sample[feature] == node['value']:
        return predict(node['left'], sample)

    return predict(node['right'], sample)


tree = build_tree(data, features, target)

print("Generated CART Regression Tree:")
pprint(tree, sort_dicts=False)


new_sample = pd.Series({
    'Experience': 'Medium',
    'Performance': 'Good',
    'Education': 'PG',
    'Projects': 'h'
})

prediction = predict(tree, new_sample)

print("Predicted Salary:", round(prediction, 2))

Generated CART Regression Tree:
{'Experience': {'value': 'Low',
                'left': {'Projects': {'value': 'l',
                                      'left': {'Performance': {'value': 'Average',
                                                               'left': np.float64(30000.0),
                                                               'right': {'Education': {'value': 'UG',
                                                                                       'left': np.float64(23500.0),
                                                                                       'right': np.float64(24000.0)}}}},
                                      'right': np.float64(40000.0)}},
                'right': {'Projects': {'value': 'h',
                                       'left': {'Education': {'value': 'PG',
                                                              'left': {'Performance': {'value': 'Good',
                                                                  

In [ ]:
import pandas as pd
from collections import Counter

data = [
    [1, "Sunny", 85, 85, "Weak", "No"],
    [2, "Sunny", 80, 90, "Strong", "No"],
    [3, "Overcast", 83, 78, "Weak", "Yes"],
    [4, "Rain", 70, 96, "Weak", "Yes"],
    [5, "Rain", 68, 80, "Weak", "Yes"],
    [6, "Rain", 65, 70, "Strong", "No"],
    [7, "Overcast", 64, 65, "Strong", "Yes"],
    [8, "Sunny", 72, 95, "Weak", "No"],
    [9, "Sunny", 69, 70, "Weak", "Yes"],
    [10, "Rain", 75, 80, "Weak", "Yes"],
    [11, "Sunny", 75, 70, "Strong", "Yes"],
    [12, "Overcast", 72, 90, "Strong", "Yes"],
    [13, "Overcast", 81, 75, "Weak", "Yes"],
    [14, "Rain", 71, 80, "Strong", "No"]
]

columns = ["Day", "Outlook", "Temp", "Humidity", "Wind", "Decision"]

df = pd.DataFrame(data, columns=columns)
df.to_csv("tennis.csv", index=False)

# Read the CSV back. The tree functions use a list of dictionaries.
df = pd.read_csv("tennis.csv")
rows = df.to_dict(orient="records")

FEATURES = ["Outlook", "Temp", "Humidity", "Wind"]
NUMERIC = ["Temp", "Humidity"]


def gini(rows):
    if len(rows) == 0:
        return 0

    counts = Counter(row["Decision"] for row in rows)
    total = len(rows)
    result = 1

    for count in counts.values():
        probability = count / total
        result -= probability ** 2

    return result


def possible_values(rows, feature):
    values = sorted({row[feature] for row in rows})

    if feature not in NUMERIC:
        return values

    thresholds = []

    for i in range(len(values) - 1):
        midpoint = (values[i] + values[i + 1]) / 2
        thresholds.append(midpoint)

    return thresholds


def split_rows(rows, feature, value):
    left = []
    right = []

    for row in rows:
        if feature in NUMERIC:
            goes_left = row[feature] <= value
        else:
            goes_left = row[feature] == value

        if goes_left:
            left.append(row)
        else:
            right.append(row)

    return left, right


def find_best_split(rows):
    best_split = None
    best_score = float("inf")

    for feature in FEATURES:
        values = possible_values(rows, feature)

        for value in values:
            left, right = split_rows(rows, feature, value)

            if len(left) == 0 or len(right) == 0:
                continue

            total = len(rows)

            score = (
                (len(left) / total) * gini(left)
                + (len(right) / total) * gini(right)
            )

            if score < best_score:
                best_score = score
                best_split = (feature, value, left, right)

    return best_split, best_score


def build_tree(rows):
    counts = Counter(row["Decision"] for row in rows)
    majority = counts.most_common(1)[0][0]

    if len(counts) == 1:
        return majority

    best_split, best_score = find_best_split(rows)

    if best_split is None or best_score >= gini(rows):
        return majority

    feature, value, left, right = best_split

    tree = {
        "feature": feature,
        "value": value,
        "fallback": majority,
        "left": build_tree(left),
        "right": build_tree(right)
    }

    return tree



def predict(tree, sample):
    current = tree

    while isinstance(current, dict):
        feature = current["feature"]
        value = current["value"]

        if feature not in sample:
            return current["fallback"]

        if feature in NUMERIC:
            goes_left = sample[feature] <= value
        else:
            goes_left = sample[feature] == value

        if goes_left:
            current = current["left"]
        else:
            current = current["right"]

    return current

tree = build_tree(rows)

from pprint import pprint
print("CART decision tree:")
pprint(tree,sort_dicts=False)

new_day = {
    "Outlook": "Sunny",
    "Temp": 74,
    "Humidity": 70,
    "Wind": "Weak"
}

print("\nNew sample:", new_day)
print("Predicted Play Tennis:", predict(tree, new_day))

correct = 0

for row in rows:
    if predict(tree, row) == row["Decision"]:
        correct += 1

print("Training rows classified correctly:", correct, "/", len(rows))

CART decision tree:
{'feature': 'Outlook',
 'value': 'Overcast',
 'fallback': 'Yes',
 'left': 'Yes',
 'right': {'feature': 'Temp',
           'value': 77.5,
           'fallback': 'No',
           'left': {'feature': 'Temp',
                    'value': 66.5,
                    'fallback': 'Yes',
                    'left': 'No',
                    'right': {'feature': 'Temp',
                              'value': 70.5,
                              'fallback': 'Yes',
                              'left': 'Yes',
                              'right': {'feature': 'Temp',
                                        'value': 73.5,
                                        'fallback': 'No',
                                        'left': 'No',
                                        'right': 'Yes'}}},
           'right': 'No'}}

New sample: {'Outlook': 'Sunny', 'Temp': 74, 'Humidity': 70, 'Wind': 'Weak'}
Predicted Play Tennis: Yes
Training rows classified correctly: 14 / 14
